<a href="https://colab.research.google.com/github/NINE4zzzz/text-and-handwriting-cnn/blob/main/%D0%B0%D1%82%D1%82%D0%B5%D1%81%D1%82%D0%B0%D1%86%D0%B8%D1%8F_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Промежуточная аттестация № 3
## Нейронные сети: классификация текста и распознавание рукописных символов

**Цель работы:** решить две задачи классификации с помощью нейронных сетей и сравнить результат с базовым алгоритмом там, где это уместно.

1. Построить бинарный классификатор текста: человек (`0`) или ИИ (`1`) на датасете [AI vs Human Text](https://www.kaggle.com/datasets/shanegerami/ai-vs-human-text).
2. Создать сверточную нейронную сеть для распознавания рукописных цифр `0–9` из MNIST и математических знаков `+` и `−` из HASYv2. На выходе будет единая модель с 12 классами, пригодная для подключения к ассистенту.

В ноутбуке есть исследование данных, графики, обучение, оценка качества, примеры предсказаний и сохранение моделей. Все итоговые метрики рассчитываются при запуске ячеек — они намеренно не подставлены заранее.

> **Перед запуском:** рекомендуется Google Colab с GPU. Для загрузки датасетов с Kaggle нужен аккаунт Kaggle и API-токен `kaggle.json`. В Colab ячейка ниже предложит загрузить его. Не публикуйте токен и не добавляйте его в отправляемый ноутбук.

## 1. Установка библиотек и подготовка среды

In [ ]:
%pip -q install kagglehub tensorflow scikit-learn pandas numpy matplotlib seaborn pillow

In [ ]:
import os
import re
import random
import shutil
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support, roc_curve, auc
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

import kagglehub

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0
GPU: []


## 2. Настройка доступа к Kaggle

In [ ]:
def configure_kaggle_credentials():
    """Use existing Kaggle credentials or request kaggle.json in Google Colab."""
    config_path = Path.home() / '.kaggle' / 'kaggle.json'
    local_path = Path('kaggle.json')

    if config_path.exists():
        print('Kaggle credentials found in ~/.kaggle/kaggle.json')
        return

    if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
        print('Kaggle credentials found in environment variables')
        return

    if local_path.exists():
        config_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_path, config_path)
        try:
            config_path.chmod(0o600)
        except OSError:
            pass
        print('Kaggle credentials copied to ~/.kaggle/kaggle.json')
        return

    try:
        from google.colab import files
        print('Загрузите файл kaggle.json, созданный в настройках аккаунта Kaggle.')
        uploaded = files.upload()
        if 'kaggle.json' not in uploaded:
            raise FileNotFoundError('Файл kaggle.json не был загружен.')
        config_path.parent.mkdir(parents=True, exist_ok=True)
        config_path.write_bytes(uploaded['kaggle.json'])
        try:
            config_path.chmod(0o600)
        except OSError:
            pass
        print('Токен сохранён в домашней директории текущего runtime.')
    except ImportError as exc:
        raise FileNotFoundError(
            'Не найден Kaggle API-токен. Поместите kaggle.json в текущую папку '
            'или ~/.kaggle/kaggle.json, либо задайте KAGGLE_USERNAME и KAGGLE_KEY.'
        ) from exc

configure_kaggle_credentials()

Загрузите файл kaggle.json, созданный в настройках аккаунта Kaggle.


# Часть I. Определение текста, написанного человеком или ИИ

## 3. Загрузка и первичное исследование датасета

Датасет может занимать значительный объём, поэтому CSV читается частями. Для учебного эксперимента используется ограниченная сбалансированная выборка, чтобы уменьшить требования к памяти и ускорить обучение. Это не меняет постановку задачи, но означает, что метрики относятся к выбранной выборке, а не ко всему датасету.

In [ ]:
def find_csv_with_columns(root, required_columns=('text', 'generated')):
    """Find the first CSV whose header contains the required column names."""
    root = Path(root)
    for csv_path in root.rglob('*.csv'):
        try:
            header = pd.read_csv(csv_path, nrows=0, encoding='utf-8-sig')
            normalized = {str(col).replace('\ufeff', '').strip().lower(): col for col in header.columns}
            if all(name in normalized for name in required_columns):
                return csv_path, {name: normalized[name] for name in required_columns}
        except Exception:
            continue
    raise FileNotFoundError(
        f'Не найден CSV с колонками {required_columns} в папке {root}. '
        'Проверьте структуру скачанного датасета.'
    )

text_data_dir = Path(kagglehub.dataset_download('shanegerami/ai-vs-human-text'))
print('Папка датасета:', text_data_dir)
text_csv_path, text_columns = find_csv_with_columns(text_data_dir, ('text', 'generated'))
print('CSV:', text_csv_path)
print('Колонки:', text_columns)

In [ ]:
def encode_generated_label(series):
    """Normalize common encodings: 0=human, 1=AI/generated."""
    numeric = pd.to_numeric(series, errors='coerce')
    if numeric.notna().mean() > 0.98 and set(numeric.dropna().unique()).issubset({0, 1}):
        return numeric.astype('float32')

    value_map = {
        '0': 0, 'human': 0, 'human-written': 0, 'human written': 0,
        'human generated': 0, 'false': 0,
        '1': 1, 'ai': 1, 'generated': 1, 'ai-generated': 1,
        'ai generated': 1, 'machine': 1, 'true': 1,
    }
    normalized = series.astype(str).str.strip().str.lower()
    mapped = normalized.map(value_map)
    mapped = mapped.where(mapped.notna(), normalized.map(
        lambda x: 0 if 'human' in x else (1 if ('ai' in x or 'generated' in x or 'machine' in x) else np.nan)
    ))
    return mapped.astype('float32')


def load_balanced_text_sample(csv_path, text_col, label_col,
                              max_per_class=10_000, chunk_size=50_000,
                              sample_per_class_per_chunk=2_500):
    """Stream a large CSV and keep a reproducible, approximately balanced sample."""
    parts = []
    for chunk_number, chunk in enumerate(pd.read_csv(
        csv_path, usecols=[text_col, label_col], chunksize=chunk_size,
        encoding='utf-8-sig', on_bad_lines='skip', low_memory=False
    )):
        labels = encode_generated_label(chunk[label_col])
        part = pd.DataFrame({
            'text': chunk[text_col].astype(str),
            'label': labels
        }).dropna(subset=['label'])
        part = part[part['label'].isin([0, 1])]
        part = part[part['text'].str.strip().str.len() > 20]

        for label in (0, 1):
            class_part = part[part['label'] == label]
            if len(class_part) > sample_per_class_per_chunk:
                class_part = class_part.sample(
                    sample_per_class_per_chunk, random_state=SEED + chunk_number + label
                )
            if len(class_part):
                parts.append(class_part)

        print(f'Обработан блок {chunk_number + 1}; строк в блоке: {len(chunk):,}')

    if not parts:
        raise ValueError('Не удалось получить примеры. Проверьте названия столбцов и метки.')

    sample = pd.concat(parts, ignore_index=True)
    sample['text'] = sample['text'].str.strip()
    sample = sample.drop_duplicates(subset=['text']).reset_index(drop=True)

    balanced_parts = []
    for label in (0, 1):
        group = sample[sample['label'] == label]
        if len(group) > max_per_class:
            group = group.sample(max_per_class, random_state=SEED + label)
        balanced_parts.append(group)

    result = pd.concat(balanced_parts, ignore_index=True)
    result['label'] = result['label'].astype('int32')
    result = result.sample(frac=1, random_state=SEED).reset_index(drop=True)
    if result['label'].nunique() != 2:
        raise ValueError('После фильтрации остался только один класс.')
    return result

text_df = load_balanced_text_sample(
    text_csv_path,
    text_col=text_columns['text'],
    label_col=text_columns['generated'],
    max_per_class=10_000,
)

print('\nРазмер выборки:', text_df.shape)
print('0 = человек, 1 = ИИ')
display(text_df.head(3))
display(text_df['label'].value_counts().sort_index().rename(index={0: 'Человек', 1: 'ИИ'}).to_frame('Количество'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
text_df['label'].value_counts().sort_index().rename(index={0: 'Человек', 1: 'ИИ'}).plot(
    kind='bar', ax=axes[0], title='Распределение классов'
)
axes[0].set_xlabel('Класс')
axes[0].set_ylabel('Количество текстов')
axes[0].tick_params(axis='x', rotation=0)

text_df['word_count'] = text_df['text'].str.split().str.len()
for label, name in [(0, 'Человек'), (1, 'ИИ')]:
    axes[1].hist(text_df.loc[text_df['label'] == label, 'word_count'].clip(upper=1500),
                 bins=40, alpha=0.55, label=name)
axes[1].set_title('Длина текстов по количеству слов')
axes[1].set_xlabel('Количество слов (значения выше 1500 ограничены на графике)')
axes[1].set_ylabel('Частота')
axes[1].legend()
plt.tight_layout()
plt.show()

print('Средняя длина в словах по классам:')
display(text_df.groupby('label')['word_count'].agg(['count', 'mean', 'median', 'max']).rename(index={0:'Человек', 1:'ИИ'}))

## 4. Разделение выборки и нейросетевая модель

Тексты делятся на обучающую, валидационную и тестовую части. Векторизатор `TextVectorization` строит словарь только по обучающим текстам. Далее используется embedding-слой и одномерная свёртка (`Conv1D`), после которых сеть выдаёт вероятность класса «ИИ».

In [ ]:
X_text = text_df['text'].astype(str)
y_text = text_df['label'].astype('int32')

X_train_full, X_test_text, y_train_full, y_test_text = train_test_split(
    X_text, y_text, test_size=0.20, random_state=SEED, stratify=y_text
)
X_train_text, X_val_text, y_train_text, y_val_text = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=SEED, stratify=y_train_full
)

print(f'Обучение: {len(X_train_text):,}; валидация: {len(X_val_text):,}; тест: {len(X_test_text):,}')

MAX_TOKENS = 30_000
SEQUENCE_LENGTH = 256
BATCH_SIZE_TEXT = 128

tokenizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode='int',
    output_sequence_length=SEQUENCE_LENGTH,
    standardize='lower_and_strip_punctuation',
)
tokenizer.adapt(tf.data.Dataset.from_tensor_slices(X_train_text.to_numpy(dtype=str)).batch(512))
actual_vocab_size = len(tokenizer.get_vocabulary())
print('Размер словаря:', actual_vocab_size)

text_inputs = keras.Input(shape=(), dtype=tf.string, name='text')
x = tokenizer(text_inputs)
x = layers.Embedding(input_dim=MAX_TOKENS, output_dim=64, mask_zero=True)(x)
x = layers.Conv1D(64, kernel_size=5, padding='same', activation='relu')(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.25)(x)
text_outputs = layers.Dense(1, activation='sigmoid', name='ai_probability')(x)
text_model = keras.Model(text_inputs, text_outputs, name='human_vs_ai_text_cnn')
text_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')],
)
text_model.summary()

In [ ]:
def make_text_dataset(texts, labels, batch_size=128, training=False):
    ds = tf.data.Dataset.from_tensor_slices((np.asarray(texts, dtype=str), np.asarray(labels, dtype=np.int32)))
    if training:
        ds = ds.shuffle(min(len(texts), 10_000), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_text_ds = make_text_dataset(X_train_text, y_train_text, training=True)
val_text_ds = make_text_dataset(X_val_text, y_val_text)
test_text_ds = make_text_dataset(X_test_text, y_test_text)

text_callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
]
text_history = text_model.fit(
    train_text_ds,
    validation_data=val_text_ds,
    epochs=5,
    callbacks=text_callbacks,
    verbose=1,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(text_history.history['accuracy'], label='train')
axes[0].plot(text_history.history['val_accuracy'], label='validation')
axes[0].set_title('Точность по эпохам — текстовая CNN')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(text_history.history['loss'], label='train')
axes[1].plot(text_history.history['val_loss'], label='validation')
axes[1].set_title('Функция потерь по эпохам — текстовая CNN')
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('Loss')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
y_text_probability = text_model.predict(X_test_text.to_numpy(dtype=str), batch_size=256, verbose=0).ravel()
y_text_pred = (y_text_probability >= 0.5).astype('int32')
text_accuracy = accuracy_score(y_test_text, y_text_pred)
text_precision, text_recall, text_f1, _ = precision_recall_fscore_support(
    y_test_text, y_text_pred, average='binary', zero_division=0
)

print(f'Тестовая accuracy: {text_accuracy:.4f}')
print(f'Precision для класса ИИ: {text_precision:.4f}')
print(f'Recall для класса ИИ: {text_recall:.4f}')
print(f'F1 для класса ИИ: {text_f1:.4f}')
print('\nПодробный отчёт:')
print(classification_report(y_test_text, y_text_pred, target_names=['Человек', 'ИИ'], digits=4, zero_division=0))

cm_text = confusion_matrix(y_test_text, y_text_pred, labels=[0, 1])
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm_text, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Человек', 'ИИ'], yticklabels=['Человек', 'ИИ'])
plt.title('Матрица ошибок — классификация текста')
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.tight_layout()
plt.show()

fpr, tpr, _ = roc_curve(y_test_text, y_text_probability)
plt.figure(figsize=(5.5, 4.5))
plt.plot(fpr, tpr, label=f'AUC = {auc(fpr, tpr):.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', label='Случайный классификатор')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-кривая — классификация текста')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Эксперимент: сравнение с TF-IDF + Logistic Regression

Нейросеть — основное решение задания. Для дополнительного эксперимента сравним её с простым классическим классификатором на тех же обучающей и тестовой частях. Сравнение помогает понять, даёт ли выбранная нейросетевая архитектура преимущество на данной выборке.

In [ ]:
tfidf = TfidfVectorizer(max_features=40_000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

baseline_model = LogisticRegression(max_iter=300, class_weight='balanced', random_state=SEED)
baseline_model.fit(X_train_tfidf, y_train_text)
y_baseline_pred = baseline_model.predict(X_test_tfidf)

baseline_accuracy = accuracy_score(y_test_text, y_baseline_pred)
_, _, baseline_f1, _ = precision_recall_fscore_support(
    y_test_text, y_baseline_pred, average='binary', zero_division=0
)

comparison_text = pd.DataFrame({
    'Модель': ['CNN + Embedding', 'TF-IDF + Logistic Regression'],
    'Accuracy': [text_accuracy, baseline_accuracy],
    'F1 (класс ИИ)': [text_f1, baseline_f1],
})
display(comparison_text.round(4))

comparison_text.set_index('Модель').plot(kind='bar', figsize=(9, 4), ylim=(0, 1), rot=0)
plt.title('Сравнение моделей на тестовой выборке')
plt.ylabel('Оценка')
plt.tight_layout()
plt.show()

if text_f1 > baseline_f1:
    print('На этой тестовой выборке CNN показала больший F1 для класса ИИ, чем базовая модель.')
elif text_f1 < baseline_f1:
    print('На этой тестовой выборке TF-IDF + Logistic Regression показала больший F1 для класса ИИ.')
else:
    print('На этой тестовой выборке F1 моделей совпал.')

### Промежуточный вывод по первой задаче

После выполнения ячеек выше получены графики обучения, матрица ошибок, ROC-кривая и сравнение с базовой моделью. При интерпретации важно смотреть не только на accuracy, но и на precision/recall: ложное определение человеческого текста как сгенерированного и пропуск ИИ-текста — разные типы ошибок. Классификатор отражает закономерности конкретного датасета; он не может достоверно установить авторство любого текста в отрыве от обучающих данных.

# Часть II. Распознавание рукописных цифр и знаков `+` и `−`

## 6. Загрузка MNIST и HASYv2

MNIST содержит изображения цифр `0–9` размером `28×28`. HASYv2 содержит изображения рукописных математических и других символов. Из HASYv2 автоматически выбираются классы «плюс» и «минус» по файлу `symbols.csv`; если конкретная версия датасета использует непривычные обозначения, ячейка покажет таблицу классов для проверки.

In [ ]:
# MNIST загружается средствами Keras.
(x_train_mnist_raw, y_train_mnist), (x_test_mnist_raw, y_test_mnist) = keras.datasets.mnist.load_data()
print('MNIST train:', x_train_mnist_raw.shape, y_train_mnist.shape)
print('MNIST test :', x_test_mnist_raw.shape, y_test_mnist.shape)

# HASYv2 скачивается из Kaggle.
hasy_root = Path(kagglehub.dataset_download('guru001/hasyv2'))
print('Папка HASYv2:', hasy_root)

label_candidates = list(hasy_root.rglob('hasy-data-labels.csv'))
symbol_candidates = list(hasy_root.rglob('symbols.csv'))
if not label_candidates:
    raise FileNotFoundError('В скачанном HASYv2 не найден файл hasy-data-labels.csv.')
if not symbol_candidates:
    raise FileNotFoundError('В скачанном HASYv2 не найден файл symbols.csv.')

hasy_labels_path = label_candidates[0]
symbols_path = symbol_candidates[0]
hasy_labels = pd.read_csv(hasy_labels_path)
symbols_df = pd.read_csv(symbols_path)
print('Файл меток:', hasy_labels_path)
print('Столбцы меток:', list(hasy_labels.columns))
print('Столбцы словаря символов:', list(symbols_df.columns))
display(symbols_df.head())

In [ ]:
def normalize_symbol_token(value):
    """Normalize LaTeX or textual symbol descriptions for matching + and - aliases."""
    if pd.isna(value):
        return ''
    value = unicodedata.normalize('NFKC', str(value)).strip().lower()
    value = value.replace('$', '')
    value = re.sub(r'\\(?:text|mathrm|mathord|ensuremath|mbox|operatorname)\s*', '', value)
    value = value.replace('{', '').replace('}', '').replace('\\', '')
    value = re.sub(r'\s+', '', value)
    return value


def normalize_symbol_id(value):
    try:
        return str(int(float(value)))
    except (ValueError, TypeError):
        return str(value).strip()

id_candidates = [c for c in symbols_df.columns if str(c).strip().lower() in {'symbol_id', 'symbolid', 'id'}]
if not id_candidates:
    raise KeyError(f'Не найден столбец symbol_id в symbols.csv. Колонки: {list(symbols_df.columns)}')
symbol_id_col = id_candidates[0]

preferred_description_cols = [
    c for c in symbols_df.columns if c != symbol_id_col and any(
        term in str(c).lower() for term in ['latex', 'symbol', 'unicode', 'char', 'name', 'description', 'label', 'ascii', 'math']
    )
]
if not preferred_description_cols:
    preferred_description_cols = [c for c in symbols_df.columns if c != symbol_id_col]

plus_aliases = {'+', 'plus', 'plussign', 'addition', 'additionsign', 'u+002b', '0x2b', '002b', '43'}
minus_aliases = {'-', '−', 'minus', 'minussign', 'hyphenminus', 'u+002d', '0x2d', '002d', 'u+2212', '0x2212', '2212', '45', '8722'}

plus_symbol_ids, minus_symbol_ids = [], []
for _, row in symbols_df.iterrows():
    tokens = [normalize_symbol_token(row[c]) for c in preferred_description_cols]
    if any(token in plus_aliases for token in tokens):
        plus_symbol_ids.append(normalize_symbol_id(row[symbol_id_col]))
    if any(token in minus_aliases for token in tokens):
        minus_symbol_ids.append(normalize_symbol_id(row[symbol_id_col]))

plus_symbol_ids = sorted(set(plus_symbol_ids))
minus_symbol_ids = sorted(set(minus_symbol_ids))
print('Кандидаты symbol_id для +:', plus_symbol_ids)
print('Кандидаты symbol_id для −:', minus_symbol_ids)

if not plus_symbol_ids or not minus_symbol_ids:
    print('Автоматический поиск не нашёл оба класса. Просмотрите строки метаданных, где встречаются plus/minus/+/-:')
    mask = symbols_df.astype(str).apply(
        lambda col: col.str.lower().str.contains(r'plus|minus|addition|\\\+|\\-|−|\+', regex=True, na=False)
    ).any(axis=1)
    display(symbols_df.loc[mask].head(50))
    raise ValueError(
        'Не удалось автоматически сопоставить классы + и −. По таблице symbols_df выше найдите '
        'symbol_id этих двух символов и добавьте их в plus_symbol_ids / minus_symbol_ids вручную.'
    )

In [ ]:
label_col_candidates = {str(c).strip().lower(): c for c in hasy_labels.columns}
if 'symbol_id' not in label_col_candidates:
    raise KeyError(f'Не найден symbol_id в hasy-data-labels.csv. Колонки: {list(hasy_labels.columns)}')
if 'path' not in label_col_candidates:
    raise KeyError(f'Не найден path в hasy-data-labels.csv. Колонки: {list(hasy_labels.columns)}')

hasy_id_col = label_col_candidates['symbol_id']
hasy_path_col = label_col_candidates['path']
hasy_labels['_normalized_id'] = hasy_labels[hasy_id_col].map(normalize_symbol_id)

hasy_image_dirs = [p for p in hasy_root.rglob('hasy-data') if p.is_dir()]
if not hasy_image_dirs:
    # Fallback: find a directory with many PNG files.
    hasy_image_dirs = [p.parent for p in hasy_root.rglob('*.png')]
if not hasy_image_dirs:
    raise FileNotFoundError('Не удалось найти изображения PNG в HASYv2.')

def count_pngs(folder):
    return sum(1 for _ in Path(folder).glob('*.png'))

images_dir = max(set(hasy_image_dirs), key=count_pngs)
print('Папка изображений:', images_dir)
print('PNG в папке:', count_pngs(images_dir))

operator_id_to_target = {sid: 10 for sid in plus_symbol_ids}
operator_id_to_target.update({sid: 11 for sid in minus_symbol_ids})
operator_rows = hasy_labels[hasy_labels['_normalized_id'].isin(operator_id_to_target)].copy()
operator_rows['target'] = operator_rows['_normalized_id'].map(operator_id_to_target).astype(int)

MAX_OPERATOR_IMAGES_PER_CLASS = 5_000
sampled_operator_rows = []
for target in (10, 11):
    group = operator_rows[operator_rows['target'] == target]
    if len(group) > MAX_OPERATOR_IMAGES_PER_CLASS:
        group = group.sample(MAX_OPERATOR_IMAGES_PER_CLASS, random_state=SEED + target)
    sampled_operator_rows.append(group)
operator_rows = pd.concat(sampled_operator_rows, ignore_index=True)
print('Найдено изображений + и − до чтения:', len(operator_rows))
display(operator_rows['target'].value_counts().sort_index().rename(index={10: '+', 11: '−'}).to_frame('Количество'))

In [ ]:
def load_hasy_image(image_path, size=(28, 28)):
    """Load a HASY image, put ink on dark background where possible, resize to MNIST size."""
    with Image.open(image_path) as source:
        image = source.convert('L')
    arr = np.asarray(image, dtype=np.uint8)
    border = np.concatenate([arr[0, :], arr[-1, :], arr[:, 0], arr[:, -1]])
    # Most handwritten samples have a uniform background at the image border.
    if float(np.median(border)) > 127:
        arr = 255 - arr
    image = Image.fromarray(arr).resize(size, Image.Resampling.LANCZOS)
    return np.asarray(image, dtype=np.float32) / 255.0

operator_images, operator_targets, operator_paths = [], [], []
missing_images = 0
for _, row in operator_rows.iterrows():
    image_path = images_dir / Path(str(row[hasy_path_col])).name
    if not image_path.exists():
        missing_images += 1
        continue
    try:
        operator_images.append(load_hasy_image(image_path))
        operator_targets.append(int(row['target']))
        operator_paths.append(str(image_path))
    except Exception:
        missing_images += 1

X_operator = np.asarray(operator_images, dtype=np.float32)[..., np.newaxis]
y_operator = np.asarray(operator_targets, dtype=np.int32)
print('Изображений успешно прочитано:', len(X_operator))
print('Пропущено/не прочитано:', missing_images)
print('Итоговые размеры:', X_operator.shape, y_operator.shape)
if len(np.unique(y_operator)) != 2 or min(np.bincount(y_operator, minlength=12)[10:12]) < 10:
    raise ValueError('Недостаточно изображений одного из операторов +/−. Проверьте сопоставление symbol_id.')

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for target, row_idx in [(10, 0), (11, 1)]:
    indices = np.where(y_operator == target)[0][:6]
    for ax, idx in zip(axes[row_idx], indices):
        ax.imshow(X_operator[idx, :, :, 0], cmap='gray')
        ax.set_title('+' if target == 10 else '−')
        ax.axis('off')
plt.suptitle('Примеры символов из HASYv2 после предобработки')
plt.tight_layout()
plt.show()

## 7. Подготовка общей выборки из 12 классов

Цифры MNIST и символы HASYv2 отличаются исходным разрешением. Изображения HASYv2 приводятся к `28×28`, а яркость фона нормализуется так, чтобы рисунок по возможности был светлым на тёмном фоне, как в MNIST. Классы модели: `0, 1, ..., 9, +, −`. Для разделения HASYv2 используется стратифицированное разбиение, чтобы оба оператора присутствовали и в обучении, и в тесте.

In [ ]:
# MNIST: нормализация в диапазон [0, 1] и добавление канала изображения.
x_train_mnist = (x_train_mnist_raw.astype('float32') / 255.0)[..., np.newaxis]
x_test_mnist = (x_test_mnist_raw.astype('float32') / 255.0)[..., np.newaxis]
y_train_mnist = y_train_mnist.astype('int32')
y_test_mnist = y_test_mnist.astype('int32')

# Ограничиваем число обучающих примеров каждой цифры для более равномерного объединённого набора.
rng = np.random.default_rng(SEED)
mnist_indices = []
MAX_MNIST_PER_DIGIT = 6_000
for digit in range(10):
    indices = np.flatnonzero(y_train_mnist == digit)
    take = min(len(indices), MAX_MNIST_PER_DIGIT)
    mnist_indices.extend(rng.choice(indices, size=take, replace=False).tolist())
mnist_indices = np.asarray(mnist_indices)
rng.shuffle(mnist_indices)

X_digits_train = x_train_mnist[mnist_indices]
y_digits_train = y_train_mnist[mnist_indices]
X_digits_test = x_test_mnist
y_digits_test = y_test_mnist

X_op_train, X_op_test, y_op_train, y_op_test = train_test_split(
    X_operator, y_operator, test_size=0.20, random_state=SEED, stratify=y_operator
)

X_train_all = np.concatenate([X_digits_train, X_op_train], axis=0).astype('float32')
y_train_all = np.concatenate([y_digits_train, y_op_train], axis=0).astype('int32')
X_test_all = np.concatenate([X_digits_test, X_op_test], axis=0).astype('float32')
y_test_all = np.concatenate([y_digits_test, y_op_test], axis=0).astype('int32')

order = rng.permutation(len(X_train_all))
X_train_all, y_train_all = X_train_all[order], y_train_all[order]

CLASS_NAMES = [str(i) for i in range(10)] + ['+', '−']
print('Обучение:', X_train_all.shape, 'Тест:', X_test_all.shape)
print('Классы в обучении:')
display(pd.Series(y_train_all).value_counts().sort_index().rename(index=dict(enumerate(CLASS_NAMES))).to_frame('Количество'))

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for cls, ax in enumerate(axes.flat):
    idx = np.flatnonzero(y_train_all == cls)[0]
    ax.imshow(X_train_all[idx, :, :, 0], cmap='gray')
    ax.set_title(CLASS_NAMES[cls])
    ax.axis('off')
plt.suptitle('Пример изображения для каждого класса')
plt.tight_layout()
plt.show()

## 8. Обучение сверточной нейронной сети

Используется простая CNN в стиле LeNet: два сверточных блока, pooling, полносвязный слой и softmax-выход на 12 классов. Веса классов компенсируют разницу в количестве изображений цифр и операторов. В качестве метрик выводятся общая точность и отдельно точность на цифрах и математических знаках.

In [ ]:
input_image = keras.Input(shape=(28, 28, 1), name='handwritten_symbol')
x = layers.Conv2D(32, kernel_size=3, padding='same', activation='relu')(input_image)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(64, kernel_size=3, padding='same', activation='relu')(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.35)(x)
output_class = layers.Dense(12, activation='softmax', name='symbol_class')(x)
symbol_model = keras.Model(input_image, output_class, name='mnist_digits_plus_minus_cnn')
symbol_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
symbol_model.summary()

class_ids = np.unique(y_train_all)
class_weights_values = compute_class_weight(
    class_weight='balanced', classes=class_ids, y=y_train_all
)
class_weights = {int(class_id): float(weight) for class_id, weight in zip(class_ids, class_weights_values)}
print('Веса классов:', class_weights)

symbol_callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
]
symbol_history = symbol_model.fit(
    X_train_all, y_train_all,
    validation_split=0.10,
    epochs=5,
    batch_size=128,
    class_weight=class_weights,
    callbacks=symbol_callbacks,
    verbose=1,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(symbol_history.history['accuracy'], label='train')
axes[0].plot(symbol_history.history['val_accuracy'], label='validation')
axes[0].set_title('Точность по эпохам — CNN символов')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(symbol_history.history['loss'], label='train')
axes[1].plot(symbol_history.history['val_loss'], label='validation')
axes[1].set_title('Функция потерь по эпохам — CNN символов')
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('Loss')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
symbol_test_loss, symbol_test_accuracy = symbol_model.evaluate(X_test_all, y_test_all, verbose=0)
y_symbol_probs = symbol_model.predict(X_test_all, batch_size=256, verbose=0)
y_symbol_pred = y_symbol_probs.argmax(axis=1)

print(f'Общая точность на объединённом тесте: {symbol_test_accuracy:.4f}')
print(f'Точность на цифрах MNIST: {accuracy_score(y_test_mnist, symbol_model.predict(X_test_mnist, verbose=0).argmax(axis=1)):.4f}')
print(f'Точность на знаках HASYv2 (+/−): {accuracy_score(y_op_test, symbol_model.predict(X_op_test, verbose=0).argmax(axis=1)):.4f}')
print('\nОтчёт по всем 12 классам:')
print(classification_report(
    y_test_all, y_symbol_pred,
    labels=list(range(12)), target_names=CLASS_NAMES,
    digits=4, zero_division=0
))

cm_symbols = confusion_matrix(y_test_all, y_symbol_pred, labels=list(range(12)))
plt.figure(figsize=(10, 8))
sns.heatmap(cm_symbols, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Матрица ошибок — цифры и математические знаки')
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.tight_layout()
plt.show()

In [ ]:
# Показать примеры предсказаний на тестовой выборке.
show_count = 12
sample_indices = np.random.default_rng(SEED).choice(len(X_test_all), size=show_count, replace=False)
sample_probs = symbol_model.predict(X_test_all[sample_indices], verbose=0)
sample_preds = sample_probs.argmax(axis=1)

fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for ax, idx, pred, probs in zip(axes.flat, sample_indices, sample_preds, sample_probs):
    ax.imshow(X_test_all[idx, :, :, 0], cmap='gray')
    true_name = CLASS_NAMES[int(y_test_all[idx])]
    pred_name = CLASS_NAMES[int(pred)]
    ax.set_title(f'ист.: {true_name}\nмодель: {pred_name} ({probs[pred]:.2f})')
    ax.axis('off')
plt.suptitle('Примеры распознавания на тестовых изображениях')
plt.tight_layout()
plt.show()

## 9. Функция распознавания для подключения к ассистенту

Функция принимает путь к файлу или объект изображения PIL, применяет ту же базовую предобработку и возвращает предсказанный символ, вероятность и распределение вероятностей по классам. Для произвольных фотографий нужно учитывать, что модель обучалась на небольших изображениях символов, а не на целых страницах: перед распознаванием страницы символ следует выделить в отдельный фрагмент.

In [ ]:
def preprocess_user_image(image_or_path):
    """Prepare a single handwritten symbol image for the 12-class CNN."""
    if isinstance(image_or_path, (str, Path)):
        image = Image.open(image_or_path).convert('L')
    elif isinstance(image_or_path, Image.Image):
        image = image_or_path.convert('L')
    else:
        raise TypeError('Передайте путь к файлу или объект PIL.Image.')

    arr = np.asarray(image, dtype=np.uint8)
    border = np.concatenate([arr[0, :], arr[-1, :], arr[:, 0], arr[:, -1]])
    if float(np.median(border)) > 127:
        arr = 255 - arr
    arr = np.asarray(Image.fromarray(arr).resize((28, 28), Image.Resampling.LANCZOS), dtype=np.float32) / 255.0
    return arr[np.newaxis, ..., np.newaxis]


def recognize_symbol(image_or_path):
    """Predict a handwritten digit (0-9) or operator (+ / −)."""
    prepared = preprocess_user_image(image_or_path)
    probabilities = symbol_model.predict(prepared, verbose=0)[0]
    index = int(np.argmax(probabilities))
    return {
        'symbol': CLASS_NAMES[index],
        'confidence': float(probabilities[index]),
        'probabilities': {name: float(prob) for name, prob in zip(CLASS_NAMES, probabilities)},
    }

# Пример: проверяем функцию на первом тестовом изображении HASYv2.
example_image = Image.fromarray(np.uint8(np.clip(X_op_test[0, :, :, 0] * 255, 0, 255)))
display(example_image.resize((140, 140)))
print(recognize_symbol(example_image))

## 10. Сохранение моделей и автоматические выводы

После выполнения ноутбука в рабочей папке появятся два файла: нейросеть для текста и CNN для распознавания цифр/операторов. Их можно загрузить в отдельном приложении через `keras.models.load_model(...)`. Перед переносом в production необходимо проверить качество на собственных данных и обеспечить обработку ошибок.

In [ ]:
text_model_path = Path('human_ai_text_cnn.keras')
symbol_model_path = Path('handwritten_digits_plus_minus_cnn.keras')
text_model.save(text_model_path)
symbol_model.save(symbol_model_path)
print('Сохранены модели:')
print('-', text_model_path.resolve())
print('-', symbol_model_path.resolve())

In [ ]:
print('ИТОГОВЫЕ РЕЗУЛЬТАТЫ')
print('=' * 60)
print(f'Задача 1 — CNN классификации текста: accuracy={text_accuracy:.4f}, F1(ИИ)={text_f1:.4f}')
print(f'Задача 1 — TF-IDF + Logistic Regression: accuracy={baseline_accuracy:.4f}, F1(ИИ)={baseline_f1:.4f}')
print(f'Задача 2 — CNN 12 классов: общая тестовая accuracy={symbol_test_accuracy:.4f}')
print('Классы модели распознавания:', ', '.join(CLASS_NAMES))

best_text_model = 'CNN + Embedding' if text_f1 >= baseline_f1 else 'TF-IDF + Logistic Regression'
print(f'По F1 для класса ИИ на этой выборке лучший результат показала: {best_text_model}.')
print('Вывод: обе задачи сведены к классификации; модели оценены на отложенных тестовых данных, а графики и матрицы ошибок позволяют проверить характер ошибок.')

## Заключение

В работе реализованы две системы классификации:

- **Классификация текста:** нейронная сеть обрабатывает текстовые последовательности и определяет класс «человек/ИИ». Дополнительно выполнено сравнение с TF-IDF + Logistic Regression. Основные критерии оценки — accuracy и F1 для класса ИИ; ROC-кривая показывает компромисс между полнотой и ложноположительными срабатываниями.
- **Распознавание символов:** CNN в стиле LeNet обучается на объединённых данных MNIST и HASYv2 и предсказывает один из 12 классов: цифры `0–9`, `+`, `−`. Качество оценивается общей точностью, отчётом по каждому классу и матрицей ошибок. Функция `recognize_symbol()` демонстрирует интерфейс, который можно подключить к ассистенту.

**Ограничения эксперимента.** Для первой задачи использована сбалансированная подвыборка крупного датасета; для HASYv2 — ограниченное число изображений двух операторов. Точность зависит от состава данных, предобработки, случайного разбиения и вычислительных ресурсов. Высокая метрика на этих тестовых выборках не гарантирует аналогичное качество на новых источниках текста или рукописных изображениях. Классификатор текста не следует трактовать как доказательство авторства, а распознавание символов на целых страницах требует отдельного этапа выделения символов.

## Источники данных и документация

1. [AI vs Human Text — Kaggle](https://www.kaggle.com/datasets/shanegerami/ai-vs-human-text).
2. [HASYv2 — Kaggle](https://www.kaggle.com/datasets/guru001/hasyv2).
3. [HASY dataset: структура данных и файлы меток](https://github.com/MartinThoma/HASY).
4. [MNIST — документация Keras](https://keras.io/api/datasets/mnist/).